Econometrics Project

In [ ]:
from pandemicEffectFcts import *
from garchVariationModels import *

In [ ]:
# %% ------------------------------ Pandemic Effect on Volatility ------------------------------
# 1) Download once
assets_raw = fetch_assets(TICKERS, start="2015-01-01", end="2024-12-31")

# 2) Compute metrics per asset
metrics = {asset: compute_metrics_df(df) for asset, df in assets_raw.items()}

# 3) Split into named periods
metrics_by_asset = {asset: split_by_period(mdf, PERIODS) for asset, mdf in metrics.items()}

# 4) Plots (grid per period + overlay)
plot_returns_grid(metrics_by_asset)
plot_overlays(metrics_by_asset, PERIODS)

# 5) Volatility table (your original definition)
vol_table = summarize_volatilities(metrics_by_asset)
print("\nVolatility (Std of SqLogReturn):\n", vol_table, "\n")

# 6) Assumptions diagnostics (one row per asset×period)
diag_rows = [
    check_assumptions(f"{asset} {pname}",
                      ret_series=parts[pname]["LogReturn"].to_numpy(),
                      sq_series=parts[pname]["SqLogReturn"].to_numpy())
    for asset, parts in metrics_by_asset.items()
    for pname in parts.keys()
]
assumption_df = pd.DataFrame(diag_rows)
pd.set_option("display.float_format", lambda v: f"{v:,.6g}")
print("=== Assumption checks for HAC test (squared log-returns) ===")
print(assumption_df[
    ["series","n",
     "Mean_LogRet","Var_LogRet","z_mean0","p_mean0",
     "ADF_stat","ADF_p",
     #"KPSS_stat","KPSS_p",
     "LB_p_lag5","LB_p_lag10","LB_p_lag20",
     "LRV_NW","RollVar_CV_126",
     "ZeroVar_Returns","ZeroVar_SqReturns"]
])

# 7) HAC/Newey–West robust comparisons
def series(metrics_by_asset: Dict[str, Dict[str, pd.DataFrame]], asset: str, period: str) -> np.ndarray:
    return metrics_by_asset[asset][period]["SqLogReturn"].to_numpy()

res_rows = []
# TIME effects per asset
for asset in metrics_by_asset:
    res_rows.append(safe_hac(series(metrics_by_asset, asset, "2020–2024"),
                             series(metrics_by_asset, asset, "2015–2019"),
                             f"TIME {asset}: mean(post) - mean(pre)"))

# CROSS-asset within each period
for pname in [p.name for p in PERIODS]:
    a, b = list(metrics_by_asset.keys())
    res_rows.append(safe_hac(series(metrics_by_asset, a, pname),
                             series(metrics_by_asset, b, pname),
                             f"CROSS {pname}: mean({a}) - mean({b})"))

summary_df = pd.DataFrame(res_rows)[
    ["Test","n1","n2","mean_x","mean_y","diff","HAC_SE","z_stat","p_value","lags_x","lags_y"]
]
print("\n=== HAC/Newey–West robust tests on mean volatility (squared log-returns) ===")
print(summary_df)


Papers for this first part: 
1. ADF test
Dickey & Fuller (1979)
“Distribution of the Estimators for Autoregressive Time Series with a Unit Root”
Test for unit root (stationarity).
https://www.jstor.org/stable/2286348?seq=1

2. Newey & West (1987)
“A Simple, Positive Semi-Definite, Heteroskedasticity and Autocorrelation Consistent Covariance Matrix”
Theoretic basis for test HAC-two-sample.
https://www.jstor.org/stable/1913610?seq=1

REMARK FOR THE PREVIOUS PART: when we do the variance test between the two time series pre and post covid, we can say that the data at the first date of post is uncorrelated with the data at the last date of the pre covid because in the acf test the legday is small enough

I test HAC/Newey–West mostrano in modo robusto che la volatilità dei rendimenti è aumentata significativamente dopo l’inizio della pandemia sia per Copper che per BCOM. Inoltre, Copper risulta sistematicamente più volatile rispetto al BCOM sia nel periodo pre-pandemico sia in quello post-pandemico, con un ampliamento della differenza dopo il 2020.
Le assunzioni richieste per il test, ovvero stazionarietà delle squared returns (ADF), assenza di varianza degenerata e presenza di autocorrelazione (Ljung–Box), risultano tutte soddisfatte. Pertanto, il test HAC rappresenta una procedura statistica appropriata per valutare l’effetto della pandemia sulla volatilità.

In [ ]:
# %%------------------------------ T-GARCH(1,1) vs GARCH(1,1) ------------------------------

tgarch_results = []
for asset, parts in metrics_by_asset.items():
    for period_name in parts.keys():
        eps = get_logreturns(metrics_by_asset, asset, period_name)
        fit_sym = fit_garch11(eps)
        fit_thr = fit_tgarch11(eps)
        lr = lr_test_tgarch_vs_garch(fit_sym, fit_thr)
        diag = tgarch_residual_diagnostics(eps, fit_thr["sigma2"]) if fit_thr.get("success") else {}

        row = {
            "Asset": asset,
            "Period": period_name,
            "nobs": len(eps),
            "GARCH_success": fit_sym["success"],
            "TGARCH_success": fit_thr["success"],
            "omega_sym": fit_sym["params"]["omega"] if fit_sym.get("success") else np.nan,
            "alpha_sym": fit_sym["params"]["alpha"] if fit_sym.get("success") else np.nan,
            "beta_sym": fit_sym["params"]["beta"] if fit_sym.get("success") else np.nan,
            "omega_thr": fit_thr["params"]["omega"] if fit_thr.get("success") else np.nan,
            "alpha_-": fit_thr["params"]["alpha_-"] if fit_thr.get("success") else np.nan,
            "alpha_+": fit_thr["params"]["alpha_+"] if fit_thr.get("success") else np.nan,
            "beta_thr": fit_thr["params"]["beta"] if fit_thr.get("success") else np.nan,
            "stationary_sym": fit_sym.get("stationary", False),
            "stationary_thr": fit_thr.get("stationary", False),
            "LR": lr["LR"],
            "LR_pvalue": lr["pvalue"],
        }
        row.update(diag)
        tgarch_results.append(row)

tgarch_df = pd.DataFrame(tgarch_results)
print("\n=== T-GARCH(1,1) vs GARCH(1,1) results and LR test (H0: α_- = α_+) ===")
print(tgarch_df[
    ["Asset","Period","nobs",
     "omega_sym","alpha_sym","beta_sym",
     "omega_thr","alpha_-","alpha_+","beta_thr",
     "stationary_sym","stationary_thr",
     "LR","LR_pvalue",
     "mean_z","z_stat_mean0","p_mean0",
     "LB_p_resid_lag5","LB_p_resid_lag10","LB_p_resid_lag20",
     "LB_p_sqres_lag5","LB_p_sqres_lag10","LB_p_sqres_lag20"]
])


Conclusions: T-GARCH vs GARCH and leverage effect

1. No evidence of leverage effect

For all assets and periods, the likelihood-ratio test comparing the Threshold-GARCH(1,1) against the symmetric GARCH(1,1) yields:

LR p-value = 1

This means that introducing the asymmetric parameters α_- and α_+ does not improve the model’s likelihood.
The estimated T-GARCH parameters satisfy:

α_- = α_+ = 0.05

Therefore, the T-GARCH model collapses to the symmetric GARCH, and we do not reject:

H0 : α_- = α_+.

Conclusion: There is no statistical evidence of a leverage effect in any asset or period.

2. Model stationarity

In all cases, the persistence terms satisfy:

α_- + α_+ + β ≈ 1.

This indicates high volatility persistence, typical of daily financial returns.
The estimates lie at the boundary of the stationarity region but remain acceptable for QML estimation.

3. Residual diagnostics

- Standardized residuals z_t have mean close to zero with high p-values → consistent with model assumptions.
- No significant autocorrelation in z_t → good.

However:

- For BCOM (2020–2024) the Ljung–Box tests on squared residuals z_t^2 show extremely small p-values (10^-9 to 10^-12).

This indicates remaining ARCH effects that the GARCH(1,1) structure does not fully capture.

4. Overall assessment

- The symmetric GARCH(1,1) model is adequate.
- There is no detectable asymmetry or leverage effect.
- For BCOM, especially post-2020, volatility exhibits additional clustering, suggesting that more flexible models (EGARCH, GJR-GARCH, higher-order GARCH) may provide a better fit.

T-GARCH does not detect leverage effects.
This is not unusual: T-GARCH models asymmetry in levels (ε²), which is a coarse specification and often unable to detect subtle asymmetric dynamics.
Reference: Zakoian (1994), Glosten–Jagannathan–Runkle (1993).
https://onlinelibrary.wiley.com/doi/10.1111/j.1540-6261.1993.tb05128.x

In [ ]:
# %%------------------------------ EGARCH(1,1) Symmetric vs Asymmetric ------------------------------

egarch_results = []
for asset, parts in metrics_by_asset.items():
    for period_name in parts.keys():
        eps = get_logreturns(metrics_by_asset, asset, period_name)

        fit_sym_eg = fit_egarch11_symmetric(eps)
        fit_asym_eg = fit_egarch11_asym(eps)
        lr_eg = lr_test_egarch_sym_vs_asym(fit_sym_eg, fit_asym_eg)

        diag_keys = [
                            "mean_z","z_stat_mean0","p_mean0",
                            "LB_p_resid_lag5","LB_p_resid_lag10","LB_p_resid_lag20",
                            "LB_p_sqres_lag5","LB_p_sqres_lag10","LB_p_sqres_lag20",
                        ]

        diag_eg = {k: np.nan for k in diag_keys}
        if fit_asym_eg.get("success"):
            diag_eg.update(tgarch_residual_diagnostics(eps, fit_asym_eg["sigma2"]))

        row = {
            "Asset": asset,
            "Period": period_name,
            "nobs": len(eps),
            "EG_sym_success": fit_sym_eg["success"],
            "EG_asym_success": fit_asym_eg["success"],
            "c_sym": fit_sym_eg["params"]["c"] if fit_sym_eg.get("success") else np.nan,
            "alpha_sym": fit_sym_eg["params"]["alpha"] if fit_sym_eg.get("success") else np.nan,
            "gamma_sym": fit_sym_eg["params"]["gamma"] if fit_sym_eg.get("success") else np.nan,
            "c_asym": fit_asym_eg["params"]["c"] if fit_asym_eg.get("success") else np.nan,
            "alpha_asym": fit_asym_eg["params"]["alpha"] if fit_asym_eg.get("success") else np.nan,
            "gamma_asym": fit_asym_eg["params"]["gamma"] if fit_asym_eg.get("success") else np.nan,
            "lambda_asym": fit_asym_eg["params"]["lambda"] if fit_asym_eg.get("success") else np.nan,
            "stationary_sym": fit_sym_eg.get("stationary", False),
            "stationary_asym": fit_asym_eg.get("stationary", False),
            "LR_EG": lr_eg["LR"],
            "LR_EG_pvalue": lr_eg["pvalue"],
        }
        row.update(diag_eg)
        egarch_results.append(row)

egarch_df = pd.DataFrame(egarch_results)
print("\n=== EGARCH(1,1): LR test H0: λ = 0 (no leverage) ===")
print(egarch_df[
    ["Asset","Period","nobs",
     "c_sym","alpha_sym","gamma_sym",
     "c_asym","alpha_asym","gamma_asym","lambda_asym",
     "stationary_sym","stationary_asym",
     "LR_EG","LR_EG_pvalue",
     "mean_z","z_stat_mean0","p_mean0",
     "LB_p_resid_lag5","LB_p_resid_lag10","LB_p_resid_lag20",
     "LB_p_sqres_lag5","LB_p_sqres_lag10","LB_p_sqres_lag20"]
])


Interpretation:

If LR_EG_pvalue is small → the asymmetric EGARCH with λ≠0 significantly improves the likelihood → EGARCH with leverage is needed.


If LR_EG_pvalue is large → no strong evidence that λ≠0 → symmetric EGARCH is adequate, no leverage effect in this specification.

RESULT: the pvalues are all small, therefore is evidence to reject he null hypothesis. So there is need to take λ≠0 in order to account for asymmetric effect of the negative returns. This seems to contraddict what said by the Tgarch test, but it's not unusual to have this situation because the egarch model is more sensitive than the other one.

Reasons: 

EGARCH detects a clear and statistically significant leverage effect across all assets and periods.
Why does EGARCH find leverage when T-GARCH does not?
EGARCH models asymmetry in the log-variance, which is more flexible and captures shape/scale effects that T-GARCH misses.
EGARCH reacts to sign and magnitude of shocks, not just squared shocks.
This makes EGARCH particularly good at modelling:
clustered volatility spikes,
asymmetric jumps,
risk-off episodes,
high-frequency shock propagation.
Reference: Nelson (1991), Bali et al., The Econometrics of Financial Markets.
https://www.jstor.org/stable/2938260
implementato direttamente in:
forma del modello (log σ², g(Z), leverage λ),
uso di QML gaussiano,
interpretazione del parametro λ come leverage.


T-GARCH misses leverage because:
It uses a piecewise reaction based solely on ε², not ε.
It cannot distinguish between large and small negative shocks unless their magnitude differs.
Volatility clustering in commodities is often driven by sudden downside jumps, which T-GARCH models poorly.
EGARCH captures leverage because:
It works in log-space, so shocks have multiplicative rather than additive effects.
Negative returns produce larger increases in log-variance when λ < 0.
It models the direction of the shock (sign), not just its size.
This aligns with financial literature: EGARCH is known to detect even small asymmetric volatility effects that remain invisible in GJR-GARCH / T-GARCH.

Nelson, D. B. (1991) – “Conditional Heteroskedasticity in Asset Returns: A New Approach”, Econometrica 59(2), 347–370.
Introduce l’EGARCH, discute motivazioni, proprietà e idee pratiche per la calibrazione. 
JSTOR
Implementato direttamente in:
forma del modello (log σ², g(Z), leverage λ),
uso di QML gaussiano,
interpretazione del parametro λ come leverage.

Malmsten, H. & Teräsvirta, T. (2004) – “Evaluating Exponential GARCH models”.
Discute in dettaglio starting values e problemi numerici dell’EGARCH; mostra quanto le scelte iniziali siano cruciali per la convergenza. 
Non implementi il loro algoritmo specifico,
ma segui il loro tipo di raccomandazioni:
starting values coerenti con il livello di varianza,
attenzione ai problemi numerici,
uso di fallback optimizer.
https://www.econstor.eu/handle/10419/56143


Francq, C. & Zakoian, J.-M. (2010) – “GARCH Models: Structure, Statistical Inference and Financial Applications”, Wiley.
Libro standard; copre EGARCH e altre varianti con enfasi su condizioni di stazionarietà e proprietà teoriche. 
ScienceDirect
https://books.google.fr/books?hl=it&lr=&id=XaiODwAAQBAJ&oi=fnd&pg=PP2&dq=Francq,+C.+%26+Zakoian,+J.-M.+(2010)+–+“GARCH+Models:+Structure,+Statistical+Inference+and+Financial+Applications”,+Wiley.&ots=AndCE3vMp2&sig=P9NZkqvLTAVEotca9SP3jwMr3oc#v=onepage&q&f=false
Nel tuo codice si riflettono i loro risultati in modo indiretto:
condizione |γ|<1 (implicita tramite tanh),
l’uso di QML e delle condizioni di momenti,
interpretazione della stazionarietà e della positivity constraints in GARCH/T-GARCH.

Zivot, E. (2008) – “Practical Issues in the Analysis of Univariate GARCH Models”.
Paper/tutorial con molte dritte pratiche su problemi di convergenza, scaling dei dati e diagnostica, applicabili anche a EGARCH. 
UW Faculty
Se vuoi, nel prossimo messaggio possiamo:
prendere un asset e una finestra,
lanciare fit_egarch11_symmetric/asym,
https://faculty.washington.edu/ezivot/research/practicalgarchfinal.pdf
Lo spirito di Zivot è presente in:
clipping di z e log σ²,
fallback di ottimizzazione,
enfasi sui problemi di convergenza / scaling.


In [ ]:
all_rows = []
for asset, parts in metrics_by_asset.items():
    for period_name in parts.keys():
        summ = fit_and_evaluate_vol_models(metrics_by_asset, asset, period_name,
                                           smooth_window=21, annualize=252, plot=True)
        summ.insert(0, "Period", period_name)
        summ.insert(0, "Asset", asset)
        all_rows.append(summ)

perf_df = pd.concat(all_rows, axis=0, ignore_index=True)

print("\n=== Volatility prediction performance (fit once, reuse params) ===")
print(perf_df.sort_values(["Asset","Period","QLIKE"])[["Asset","Period","Model","n","MSE","QLIKE","corr"]])

Volatility Modeling and Evaluation – Discussion of Results
This section discusses the results obtained from the estimation and evaluation of GARCH-type volatility models, namely GARCH(1,1), T-GARCH(1,1), EGARCH(1,1) symmetric, and EGARCH(1,1) asymmetric. All models are estimated using de-meaned logarithmic returns and evaluated against a realized volatility proxy derived from squared log-returns.
Volatility Measure and Proxy
Volatility is modeled as the conditional standard deviation of logarithmic returns. Let
ε_t = log(S_t / S_{t-1})
denote the log-return at time t. All models estimate the conditional variance
σ_t² = E[ε_t² | F_{t−1}].
To assess model performance, a sampled (realized) volatility proxy is constructed as
RV_t = ε_t²,
which corresponds to the squared log-return. Since daily squared returns are extremely noisy, this proxy is further smoothed using a rolling-window average. Both the model-implied volatility and the sampled volatility proxy are annualized for ease of interpretation.




Interpretation of Volatility Plots
The plots comparing model-implied volatility with the sampled volatility proxy highlight several important and theoretically expected features.
First, all models clearly capture volatility clustering. Periods of high and low volatility are correctly identified, particularly during turbulent phases such as the 2020–2024 period. This confirms that the models successfully detect changes in volatility regimes.
Second, the sampled volatility proxy remains highly noisy, even after smoothing. In contrast, model-implied volatilities are noticeably smoother. This behavior is expected, as GARCH-type models estimate the conditional expectation of future variance rather than instantaneous shocks. The smoother paths therefore reflect the underlying volatility dynamics rather than idiosyncratic return realizations.
Third, all models exhibit strong persistence and gradual mean reversion in volatility, consistent with stylized facts of financial time series.
Finally, asymmetric specifications such as T-GARCH and EGARCH asymmetric display stronger reactions to large negative returns. This leverage effect is particularly evident for Copper during high-volatility periods, supporting the relevance of asymmetric volatility modeling for commodity returns.
Overall, the visual inspection confirms that the models behave as predicted by theory and that differences across specifications are economically meaningful.



Quantitative Performance Metrics
Volatility prediction performance is evaluated using three complementary measures:
Mean Squared Error (MSE)
QLIKE loss, a scale-robust and widely used loss function for volatility evaluation
Correlation between realized variance ε_t² and model-implied variance σ_t²
The results show that correlations are positive but moderate, typically ranging between 0.1 and 0.3. This outcome is entirely expected at the daily frequency, since realized variance is highly noisy due to the multiplicative innovation term z_t². Consequently, correlation is not a sufficient measure of model quality on its own.
QLIKE provides a more reliable ranking of models and is therefore the preferred criterion. Across assets and periods, EGARCH models—especially the asymmetric specification—consistently achieve the lowest QLIKE values, indicating superior volatility fit. Symmetric GARCH models perform competitively, while T-GARCH models tend to perform less well in this setting.




Economic Interpretation
The empirical results suggest that GARCH-type models successfully capture volatility persistence and regime shifts. Allowing for asymmetric responses to shocks improves volatility modeling, particularly during periods of elevated uncertainty. The relatively low correlation between squared returns and conditional variance does not imply poor performance but rather reflects the intrinsic noisiness of daily realized volatility proxies.
These findings are fully consistent with established results in the volatility modeling literature.



Final Assessment
The implementation and evaluation of the volatility models are internally consistent, theoretically sound, and empirically plausible. Volatility is correctly defined as the conditional standard deviation of logarithmic returns, and standard Gaussian QML estimation techniques are employed. The results support the conclusion that asymmetric GARCH specifications provide the most accurate description of volatility dynamics for the assets considered.